# Test a plateau schedule on the diagnostic folds

Question: can initial learning rate 0.001 retain stage detection while later reductions improve refinement?
Reuse the completed instrumented seed-42 constant-0.001 reference. Train only the scheduled candidate.
Keep original folds 01 and 04, dropout 0.2, context thirty, width sixteen, natural sampling, and budget 40/12 fixed.
Keep Adam, unweighted loss, clipping, diagnostics, and validation-macro-F1 checkpoint selection unchanged.
The budget is two new fits and at most eighty epochs.

ReduceLROnPlateau monitors validation macro-F1 in max mode. Its factor is 0.3 and its patience is three.
It reduces after four consecutive non-improving scheduler evaluations, following its initial baseline.
The absolute improvement threshold is 0.0001, cooldown is zero, and the learning-rate floor is 0.00003.
The scheduler threshold differs from the strict improvement rule used to select the best checkpoint.
Early-stopping patience remains twelve. A reduction neither resets nor extends early stopping.

These eleven held-out people form a targeted diagnostic subset, not complete development validation.
A schedule is an optimization intervention, not a guaranteed cure for overfitting or a new global default.
NLL means negative log-likelihood; lower is better. Stage indices are 0 = Wake, 1 = Light, 2 = Deep, and 3 = REM.
Read docs/transformer-plateau.md before enabling preparation or training.


## Locate the repository and declare execution switches

**What:** Find the checkout, configure logging, and separate setup, preparation, preflight, and training switches.
**Why:** Make every costly or environment-changing operation explicit. Report execution never starts optimization.


In [ ]:
import logging
import os
import shutil
import sys
from pathlib import Path

from IPython import get_ipython
from IPython.display import display

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)
logger = logging.getLogger("transformer_plateau_notebook")
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "configs/transformer-plateau-grouped.yaml").exists()),
    Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", Path.cwd() / "sleep-classifier")),
).resolve()
ROOT = Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", str(ROOT))).expanduser().resolve()
REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "experiment/transformer-plateau"  # Select the PR branch before merge
RUN_SETUP = False
RUN_PREPARATION = False
RUN_CHECK = True
RUN_TRAINING = False
logger.info("Repository: %s", ROOT)
logger.info("Setup=%s preparation=%s preflight=%s training=%s",
            RUN_SETUP, RUN_PREPARATION, RUN_CHECK, RUN_TRAINING)


## Optional environment setup

**What:** Optionally clone this branch and install locked dependency groups with uv.
**Why:** Support another machine without environment-specific notebook branches. Reused features and reference artifacts must also be available.


In [ ]:
if RUN_SETUP:
    if not ROOT.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    if shutil.which("uv") is None:
        installer = ROOT / "artifacts/uv-install.sh"
        installer.parent.mkdir(parents=True, exist_ok=True)
        !curl -LsSf https://astral.sh/uv/install.sh -o "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv download failed")
        !sh "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    %cd "{ROOT}"
    !uv sync --locked --group gru --group notebook --group dev
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Environment setup failed")
else:
    logger.info("Environment setup skipped")


## Verify the reference, fixed controls, and budget

**What:** Verify reference hashes, original participant assignments, model settings, initial learning rate, scheduler settings, and device.
**Why:** Isolate the schedule addition and reject incompatible inputs before preparation or training.
Reuse the completed instrumented constant-0.001 fits. Do not retrain the reference here.


In [ ]:
import json

import pandas as pd

from sleep_classifier.experiment_evidence import EpochEvidenceReport
from sleep_classifier.grouped_validation import GroupedValidationExperiment
from sleep_classifier.neural_training import TrainingConfig
from sleep_classifier.paired_architecture_report import PairedArchitectureReport

experiment = GroupedValidationExperiment(ROOT / "configs/transformer-plateau-grouped.yaml")
reference = GroupedValidationExperiment(ROOT / "configs/transformer-diagnostics-grouped.yaml")
plan = experiment.plan()
reference_summary = reference.output_dir / "summary.json"
if not reference_summary.exists():
    raise FileNotFoundError("The completed instrumented reference is required")
reference_saved = json.loads(reference_summary.read_text())
if reference_saved["plan"] != reference.plan():
    raise ValueError("The saved reference differs from its current fold plan")
reference_validation = PairedArchitectureReport(reference_summary, reference_summary).build()
reference_manifest = json.loads((reference.output_dir / "manifest.json").read_text())
resolved_device = str(TrainingConfig(experiment.model_settings).device())
if resolved_device != reference_manifest["device"]:
    raise ValueError("Candidate device differs from the measured reference device")
if plan["folds"] != reference_saved["plan"]["folds"]:
    raise ValueError("Candidate folds differ from the original diagnostic partitions")
if len(plan["folds"]) != 2 or plan["maximum_epochs"] != 80:
    raise ValueError("The candidate exceeds the agreed two-fit budget")
excluded = {"output_dir", "lr_scheduler"}
recipe = {key: value for key, value in experiment.model_settings.items() if key not in excluded}
reference_recipe = {key: value for key, value in reference.model_settings.items()
                    if key not in excluded}
if recipe != reference_recipe:
    raise ValueError("Another model, training, or evidence setting changed")
if reference.model_settings.get("lr_scheduler") is not None:
    raise ValueError("The reference must use no scheduler")
if reference.model_settings["learning_rate"] != 0.001:
    raise ValueError("The reference must use initial learning rate 0.001")
if experiment.model_settings["learning_rate"] != 0.001:
    raise ValueError("The candidate must retain initial learning rate 0.001")
expected_scheduler = {
    "kind": "reduce_on_plateau", "monitor": "validation_macro_f1", "mode": "max",
    "factor": 0.3, "patience": 3, "threshold": 0.0001, "threshold_mode": "abs",
    "cooldown": 0, "min_lr": 0.00003, "eps": 1.0e-8,
}
if experiment.model_settings["lr_scheduler"] != expected_scheduler:
    raise ValueError("The candidate scheduler differs from the agreed experiment")
display(pd.DataFrame(plan["folds"]).set_index("name"))
display(pd.DataFrame([{"reference_initial_lr": 0.001, "candidate_initial_lr": 0.001,
                       "new_fits": 2, "maximum_new_epochs": 80, "device": resolved_device}]))
display(pd.DataFrame([expected_scheduler]))
logger.info("Reference verified; only the plateau schedule is added")


## Prepare only the scheduled candidate

**What:** Optionally copy development features and freeze the candidate fold configurations.
**Why:** Retain original participant splits and inputs without overwriting or retraining the completed reference.
Enable preparation once for a fresh candidate output directory.


In [ ]:
if RUN_PREPARATION:
    PairedArchitectureReport(reference_summary, reference_summary).build()
    if json.loads(reference_summary.read_text())["plan"] != reference.plan():
        raise ValueError("The reference fold plan changed")
    %cd "{ROOT}"
    !uv run --locked --group gru --group notebook python -m sleep_classifier.grouped_validation \
        --config configs/transformer-plateau-grouped.yaml --prepare
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Preparation failed; inspect existing outputs before retrying")
else:
    logger.info("Candidate preparation skipped")


## Check both prepared candidates without optimization

**What:** Run each actual prepared fold through the no-gradient input and model check.
**Why:** Verify the resolved device, scaler inputs, tensor shapes, and model size before training.
The preflight does not create trained checkpoints or diagnostic epochs.


In [ ]:
if RUN_CHECK and (experiment.output_dir / "manifest.json").exists():
    %cd "{ROOT}"
    for fold in plan["folds"]:
        fold_config = experiment.output_dir / fold["name"] / "gru.yaml"
        logger.info("Checking %s without optimization", fold["name"])
        !uv run --locked --group gru --group notebook python -m sleep_classifier.neural_training \
            --config "{fold_config}" --check
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Diagnostic preflight failed")
elif RUN_CHECK:
    logger.info("Preflight pending: prepare the two folds first")
else:
    logger.info("Preflight skipped")


## Train the scheduled candidate when ready

**What:** Optionally fit two seed-42 models with the plateau schedule enabled.
**Why:** Test only the schedule while retaining the measured reference, evidence controls, and checkpoint objective.
Keep preparation disabled once its manifest exists. Existing results are refused; automatic resume is unsupported.


In [ ]:
if RUN_TRAINING:
    PairedArchitectureReport(reference_summary, reference_summary).build()
    if json.loads(reference_summary.read_text())["plan"] != reference.plan():
        raise ValueError("The reference fold plan changed")
    %cd "{ROOT}"
    !uv run --locked --group gru --group notebook python -u -m sleep_classifier.grouped_validation \
        --config configs/transformer-plateau-grouped.yaml --train
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Candidate training failed; inspect retained evidence before retrying")
else:
    logger.info("Training skipped: enable RUN_TRAINING when ready")


## Inspect reference and candidate status

**What:** Read available epoch evidence for both recipes without fitting a model.
**Why:** Distinguish a completed reference from prepared, partial, failed, or completed candidate results.
Missing measurements remain unavailable rather than becoming zero scores.


In [ ]:
reporters = {"reference": EpochEvidenceReport(reference.output_dir),
             "candidate": EpochEvidenceReport(experiment.output_dir)}
evidence_by_role = {role: reporter.build() for role, reporter in reporters.items()}
status_rows = [{"recipe": role, "study_status": evidence["status"], "fold": fold["fold"],
                "fold_status": fold["status"], "recorded_epochs": len(fold["epochs"])}
               for role, evidence in evidence_by_role.items() for fold in evidence["folds"]]
if status_rows:
    display(pd.DataFrame(status_rows).set_index(["recipe", "fold"]))
for role, evidence in evidence_by_role.items():
    if not evidence["folds"]:
        logger.info("%s is not prepared", role)


## Compare complete diagnostic curves

**What:** Plot available losses, macro-F1, gradients, and the actual learning rates used by each epoch.
**Why:** Inspect whether scheduled refinement preserves stage detection while limiting probability-loss deterioration.
Training evaluation NLL disables dropout at fixed end-of-epoch weights. Ordinary training loss includes active dropout and changing weights.
A reduced rate applies to the next epoch. The current epoch's evidence retains the rate that its optimizer updates actually used.


In [ ]:
import matplotlib.pyplot as plt

for role, reporter in reporters.items():
    if not any(fold["epochs"] for fold in evidence_by_role[role]["folds"]):
        logger.info("%s curves are pending", role)
        continue
    figure = reporter.plot_curves()
    figure.suptitle(f"{role}: "
                   f"{'constant 0.001' if role == 'reference' else 'plateau schedule from 0.001'}")
    figure.tight_layout(rect=(0, 0, 1, 0.96))
    display(figure)
    plt.close(figure)


## Verify the schedule was exercised

**What:** Read each completed candidate fit's history and display used rates, next-epoch rates, and reduction events.
**Why:** A schedule that never reduced the rate did not exercise the proposed optimization intervention.
The existing epoch-evidence curves show rates used by optimizer updates in the current epoch.
A reduction is applied only to the next epoch, not retroactively to the measured current epoch.
At the final epoch cap or an early stop, next-epoch rates are None and no further schedule advancement occurs.
Scheduler patience three means four non-improving evaluations after the initial baseline, with an absolute threshold of 0.0001.
The strict best-checkpoint comparison and stopping patience twelve remain unchanged.


In [ ]:
scheduler_rows = []
for fold in plan["folds"]:
    metrics_path = experiment.output_dir / fold["name"] / "results/metrics.json"
    if not metrics_path.exists():
        logger.info("%s scheduler history is pending", fold["name"])
        continue
    metrics = json.loads(metrics_path.read_text())
    for epoch in metrics["history"]:
        required = {"learning_rates_used", "learning_rates_next_epoch", "scheduler_reduced"}
        if not required.issubset(epoch):
            raise ValueError("Scheduled history lacks rate-use and reduction evidence")
        scheduler_rows.append({
            "fold": fold["name"], "epoch": epoch["epoch"],
            **{key: epoch[key] for key in sorted(required)},
        })
if scheduler_rows:
    display(pd.DataFrame(scheduler_rows).set_index(["fold", "epoch"]))
    for fold in plan["folds"]:
        rows = [row for row in scheduler_rows if row["fold"] == fold["name"]]
        if not rows:
            continue
        reductions = sum(row["scheduler_reduced"] for row in rows)
        logger.info("%s observed learning-rate reductions: %s", fold["name"], reductions)
        if reductions == 0:
            logger.warning("%s has not exercised an actual rate reduction", fold["name"])
else:
    logger.info("No completed scheduled history exists yet")


## Compare selected checkpoints after completion

**What:** Verify both completed runs under the strict scheduler comparison policy and display paired scores.
**Why:** Permit only the intended schedule addition and compare identical target identities, participant folds, and training devices.
Macro-F1 stays primary. Class precision, recall, and F1 expose trade-offs hidden by pooled accuracy.


In [ ]:
comparison = None
candidate_summary = experiment.output_dir / "summary.json"
if candidate_summary.exists() and json.loads(candidate_summary.read_text())["status"] == "complete":
    comparison = PairedArchitectureReport(
        reference_summary, candidate_summary, comparison="scheduler",
    ).build()
    display(pd.DataFrame(comparison["folds"]).set_index("fold"))
    score_rows = [{"recipe": role, "accuracy": comparison[role]["accuracy"],
                   "pooled_macro_f1": comparison[role]["macro_f1"]}
                  for role in ("reference", "candidate")]
    display(pd.DataFrame(score_rows).set_index("recipe"))
    class_rows = [{"recipe": role, "stage": stage, **scores}
                  for role in ("reference", "candidate")
                  for stage, scores in comparison[role]["classes"].items()]
    display(pd.DataFrame(class_rows).set_index(["recipe", "stage"]))
    destination = experiment.output_dir / "paired-comparison.json"
    destination.write_text(json.dumps(comparison, indent=2) + "\n")
    logger.info("Paired subset comparison saved: %s", destination)
else:
    logger.info("The candidate is incomplete; no paired performance conclusion is available")


## Inspect selected NLL, fitting progress, clipping, and loss swings

**What:** Display selected-checkpoint probability loss and the largest upward validation-NLL step for each available fit.
**Why:** Distinguish a useful selected model from a quieter trajectory or slower fitting.
NLL remains secondary. Its minimum does not replace macro-F1 checkpoint selection, and unequal run lengths limit maximum-jump comparisons.


In [ ]:
diagnostic_rows = []
for role, evidence in evidence_by_role.items():
    for fold in evidence["folds"]:
        epochs = fold["epochs"]
        if not epochs:
            continue
        selected = next((epoch for epoch in epochs if epoch["epoch"] == fold["best_epoch"]), None)
        jumps = [{"after_epoch": after["epoch"],
                  "delta": after["validation_nll"] - before["validation_nll"]}
                 for before, after in zip(epochs, epochs[1:])]
        largest = max(jumps, key=lambda row: row["delta"]) if jumps else None
        diagnostic_rows.append({
            "recipe": role, "fold": fold["fold"], "completed_epochs": len(epochs),
            "selected_epoch": fold["best_epoch"],
            "selected_validation_nll": selected["validation_nll"] if selected else None,
            "selected_training_evaluation_nll": selected["training_evaluation_nll"]
            if selected else None,
            "selected_clipped_step_fraction": selected["clipped_step_fraction"]
            if selected else None,
            "final_training_evaluation_nll": epochs[-1]["training_evaluation_nll"],
            "largest_upward_nll_step": max(0.0, largest["delta"]) if largest else None,
            "largest_upward_step_epoch": largest["after_epoch"]
            if largest and largest["delta"] > 0 else None,
        })
if diagnostic_rows:
    display(pd.DataFrame(diagnostic_rows).set_index(["recipe", "fold"]))
else:
    logger.info("Diagnostic scores are pending")


## Save portable local evidence reports

**What:** Regenerate each recipe's local evidence summary and any available figures through the CLI.
**Why:** Keep inspection reproducible outside Jupyter and retain partial-run evidence after a failure.
The inspector logs an unprepared path without creating it or blocking later preparation.


In [ ]:
%cd "{ROOT}"
for role, directory in (("reference", reference.output_dir), ("candidate", experiment.output_dir)):
    logger.info("Saving %s evidence report", role)
    !uv run --locked --group gru --group notebook python -m sleep_classifier.experiment_evidence \
        --output-dir "{directory}" --report
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Evidence inspection failed")


## Export the executed HTML report

**What:** Execute report cells in a fresh kernel and save one standalone HTML report.
**Why:** Share the current evidence while keeping source notebooks output-free and skipping preparation, training, setup, and recursive export.


In [ ]:
%cd "{ROOT}"
!uv run --locked --group gru --group notebook python scripts/execute_notebook.py \
    --notebook notebooks/22_transformer_plateau.ipynb
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Plateau notebook export failed")
